In [16]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import json


from tether_script import process_single_file

### Goal
* Try to decouple the velocity from the threshold parameter. When reaching faster velocities, the dt parameter becomes very small and the threshold gets very big. In the other sens, when retracting very small, the threshold becomes very small. 

* Normalize the threshold parameter by the velocity

### Get sessions and plateau analysis files
- df_sessions is the csv that contains the settings parameters for analyzing the tethers
- df_pl_data is the output data of the find_plateaus from tether_script.py

- df_merged, merges the two dataframes above, into one with the designated filename

In [17]:

# open Tether analysis summary csv to dataframe
session_file_path = '/Users/evillz/Data/article/2025_07_01_THP1_phd/sessions/tether_session_20250716_192549.csv'
session_file_path = '/Users/evillz/Data/article/final_yey/final/tether_session_20250809_160440_concat_final_copy.csv'
df_sessions = pd.read_csv(session_file_path)
#df_pl_data = pd.read_csv('/Users/evillz/Data/article/2025_07_01_THP1_phd/sessions/batch_analysis_results/detailed_plateau_data.csv')

# write function that will merge both dataframes by matching the filenames and creating a new dataframe from the merged data
# Standardize the filename column in both dataframes for merging
df_sessions['filename'] = df_sessions['file_name']
#df_pl_data['filename'] = #df_pl_data['filename']

# Merge on the standardized 'filename' column
# df_merged = pd.merge(df_sessions, df_pl_data, on='filename', how='inner')

# #iterate through file_paxrameters in df_merged and run process_single_file for each file 
# for index, row in df_merged.iterrows():
#     file_parameters = row['file_parameters']
#     parameters = json.loads(file_parameters)
#     local_file_path = row['local_file_path']

# find a filename and get column details from df_merged
# filename = 'fcurve_thp1_cell1_ret_300ums__2025.07.01_16.49.19.87.tdms'
# row = df_merged[df_merged['filename'] == filename]

# file_parameters = row['file_parameters'][0]
# parameters = json.loads(file_parameters)
# local_file_path = row['local_file_path'][0]


# # print details of filename
# print(f"Filename: {filename}")
# for k, v in parameters.items():
#     print(f"{k}: {v}")
# print(f"Local File Path: {local_file_path}")

In [18]:
# get df_merged, go through each row of file_parameters, and modify threshold parameters
def modify_thresholds(df, threshold_modifications):
    for index, row in df.iterrows():
        file_parameters = row['file_parameters']
        parameters = json.loads(file_parameters)
        
        # Modify the parameters based on the provided modifications
        for key, value in threshold_modifications.items():
            if key in parameters:
                old_value = parameters[key]
                parameters[key] = value  # Adjust the threshold
            else:
                parameters[key] = value  # Add new threshold parameter if not present       
        # Update the dataframe with modified parameters
        df.at[index, 'file_parameters'] = json.dumps(parameters)
    
    return df

# Example threshold modifications
threshold_modifications = {
    'plateau_end_remove_percent': 50,
    # 'threshold': 0.1,  # Example modification
    # 'pl_threshold': 1e-2,   # Example modification
    # 'pl_min_width_um': 0.001
}
# Apply the modifications to the merged dataframe
# updated to modify thresholds of df_sessions
df_modified = modify_thresholds(df_sessions, threshold_modifications)

# remove repeating file_name and their respective rows
df_modified = df_modified.drop_duplicates(subset=['file_name']) 


In [19]:
#  save the modified dataframe to a new CSV file
output_file = Path('/Users/evillz/Data/article/final_yey/final/60_remove/60added_test.csv')
df_modified.to_csv(output_file, index=False)    